## Baseline scores on semart using CLIP

In [1]:
%load_ext autoreload
%autoreload 2

import torch.nn.functional as F

import open_clip 
import numpy as np
import pandas as pd


from src.model import SheafMultimodalGNN
from src.utils import *
from src.data import *
# from src.ClusterData import ClusterData, ClusterLoader
from torch_geometric.data import DataLoader
from src.metrics import *

triplets = '../artistic_sheaf/data/testing_elements.json'
#triplets = '../artistic_sheaf/data/triplets_semart_test_csv.json'
loaded_data = load_json_data(triplets)[:500]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: 

Loaded 6 triplets from ../artistic_sheaf/data/testing_elements.json


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
print(f"Using device: {device}")
seed_everything(seed=42)

# Load tokenizer and preprocessing
tokenizer = open_clip.get_tokenizer('ViT-B-32')
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
# Initialize the model
model = SheafMultimodalGNN(
    latent_dim=512,
    edge_attr_dim=512,
    num_layers=3,
    step_size=1.0,
    lr=1e-4,
    test=False,
    clip_grad=True,
    device='cuda' if torch.cuda.is_available() else 'mps'
)
    
# Load checkpoint
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=43-val_loss=5.36.ckpt", map_location=device) # sheafCLIP frozen CLIP
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=23-val_loss=5.41.ckpt", map_location=device) # no sheaf frozen CLIP
checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=01-val_loss=3.50.ckpt", map_location=device) # sheafCLIP CLIP grad
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()
print()

Using device: mps



### Testing on normal dataset

In [3]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

100%|██████████| 6/6 [00:00<00:00, 50.71it/s]

6



/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
#clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
clip_images = []
clip_texts = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_images.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
        clip_texts.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
        
clip_images = torch.cat(clip_images, dim=0)
clip_texts = torch.cat(clip_texts, dim=0)

print(f"Extracted {len(clip_texts)} text embeddings, each of shape {clip_texts[0].shape}")
print(f"Extracted {len(clip_images)} image embeddings, each of shape {clip_images[0].shape}")

clip_images = clip_images.cpu().detach().numpy()
clip_texts = clip_texts.cpu().detach().numpy()

torch.Size([3, 3, 224, 224]) torch.Size([6, 77]) torch.Size([2, 6]) torch.Size([6, 77])


/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/pytorch_lightning/core/module.py:441: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


Extracted 6 text embeddings, each of shape torch.Size([512])
Extracted 6 image embeddings, each of shape torch.Size([512])


In [8]:
clip_images[:, :10]

array([[ 0.05142807,  0.04114633,  0.00303559,  0.02620758,  0.08716618,
         0.01959026,  0.02385887, -0.03608185, -0.05528297,  0.04997332],
       [ 0.04710224,  0.04325548,  0.00551004,  0.02246621,  0.09645992,
         0.02683721,  0.02370859, -0.02873219, -0.05150824,  0.04051518],
       [ 0.04710224,  0.04325548,  0.00551004,  0.02246621,  0.09645992,
         0.02683721,  0.02370859, -0.02873219, -0.05150824,  0.04051518],
       [-0.08219391,  0.00700474,  0.01653705, -0.0179923 ,  0.05614927,
         0.05226253,  0.02117584,  0.08853501,  0.0578848 , -0.07962122],
       [-0.06119064, -0.02857943,  0.00909678, -0.02310798, -0.05952208,
         0.03091938, -0.03138187,  0.08907265,  0.07834331, -0.05655991],
       [ 0.07892877,  0.04555682,  0.00882041,  0.01383297,  0.08988177,
        -0.03467753,  0.03305071, -0.03816254, -0.06953606,  0.03853735]],
      dtype=float32)

In [ ]:
# # save embeddings images and test

# np.save('data/clip_images_grad_clip.npy', clip_images)
# np.save('data/clip_texts_grad_clip.npy', clip_texts)


In [ ]:
# clip_images = np.load('data/clip_images_grad_clip.npy')
# clip_texts = np.load('data/clip_texts_grad_clip.npy')

In [6]:
# take a subset of the image embeddings and plot them with plotly interactively (in 2D using umap) showing the edge_index[0, i] on hover
import umap
import plotly.express as px
reducer = umap.UMAP()
#from sklearn.decomposition import PCA
#reducer = PCA(n_components=2)

embedding_2d = reducer.fit_transform(np.concatenate([clip_images, clip_texts], axis=0) ) # take only first 
fig = px.scatter(x=embedding_2d[:, 0], y=embedding_2d[:, 1],
                hover_data=[np.concatenate([np.arange(len(clip_images)), np.arange(len(clip_texts))], axis=0),
                            np.concatenate([test_graph_data.edge_index[0, :len(clip_images)].cpu().numpy(), test_graph_data.edge_index[1, :len(clip_texts)].cpu().numpy()], axis=0)], 
                color=['images']*(len(clip_images)) + ['text']*(len(clip_texts)))
fig.show()

/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


### Image-to-text retrieval	and Text-to-image retrieval		
r@1	r@5	r@10	

In [43]:
#loaded_data = load_json_data("data/triplets_semart_test_orig.json")#[:9914]
#print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_orig.json")

adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data) # img_to_idx only indexes one of the indexes with the same image
print(f"Adjacency matrix shape: {adj_matrix.shape}")

sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data], 
                            [t["item2"] for t in loaded_data], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")

Adjacency matrix shape: (3, 6)
Similarity matrix shape: (3, 6)


In [23]:
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

{'t2i_precision@1': tensor(1.),
 't2i_recall@1': tensor(0.6111),
 't2i_ndcg@1': tensor(1.),
 't2i_precision@5': tensor(0.4000),
 't2i_recall@5': tensor(1.),
 't2i_ndcg@5': tensor(0.9687),
 't2i_precision@10': tensor(0.3333),
 't2i_recall@10': tensor(1.),
 't2i_ndcg@10': tensor(0.9687),
 'i2t_precision@1': tensor(0.5000),
 'i2t_recall@1': tensor(0.5000),
 'i2t_ndcg@1': tensor(0.5000),
 'i2t_precision@5': tensor(0.3333),
 'i2t_recall@5': tensor(1.),
 'i2t_ndcg@5': tensor(0.7936),
 'i2t_precision@10': tensor(0.3333),
 'i2t_recall@10': tensor(1.),
 'i2t_ndcg@10': tensor(0.7936),
 'mean_precision@1': tensor(0.7500),
 'mean_recall@1': tensor(0.5556),
 'mean_ndcg@1': tensor(0.7500),
 'mean_precision@5': tensor(0.3667),
 'mean_recall@5': tensor(1.),
 'mean_ndcg@5': tensor(0.8812),
 'mean_precision@10': tensor(0.3333),
 'mean_recall@10': tensor(1.),
 'mean_ndcg@10': tensor(0.8812)}

In [24]:
print('uniformity images', uniformity(torch.tensor(clip_images)))
print('uniformity texts', uniformity(torch.tensor(clip_texts)))
print('alignment', alignment(torch.tensor(clip_images), torch.tensor(clip_texts)))

uniformity images tensor(-2.7213)
uniformity texts tensor(-2.3729)
alignment tensor(0.0940)


### Retrieval per type of relationship

In [25]:
txt_to_idx

{'Artwork painted in 1451-1500': 0,
 "Caron, who was Catherine de Medicis' official artist and whose paintings often seem to echo the royal f&#234;tes, shows here the Emperor Augustus on his knees in front of the Sibyl, who gestures towards the Virgin and Child in the heavens. The architectural setting, reminiscent of theatre decor, shows the Seine and possibly certain monuments of Paris modified by the artist's imagination on the right-hand side": 1,
 "Of the Hungarian kings St Ladislas is perhaps the one most often represented in post-medieval frescoes, altar paintings and statues. In the Middle Ages he was associated with the ideal of chivalry and the legends which gathered around him determined the manner of his representation. Thus the knightly armour and the battle-axe have become permanent attributes, in addition to the crown and the orb.The painting illustrated here, dating from the late sixteenth century, is only superficially linked with medieval portraits of St. Ladislas. Th

In [26]:
img_to_idx

{'Images/03766-031madon.jpg': 0,
 'Images/07736-augustus.jpg': 1,
 'Images/41294-10ladisl.jpg': 2}

In [44]:
adj_matrix

array([[1, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 1, 0, 1, 1]])

In [48]:
# numpy diagonal matrix with ones shape len(loaded_data) x len(loaded_data)
adj_matrix = np.eye(len(loaded_data)).astype(int)
adj_matrix

array([[1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1]])

In [45]:
sim_matrix

array([[ 0.9374316 , -0.12680955, -0.05516497, -0.03964251, -0.05903826,
        -0.0895372 ],
       [-0.21591164,  0.97224706,  0.00385709, -0.0119141 ,  0.6894938 ,
         0.64253545],
       [-0.08548889,  0.7781937 , -0.01974309, -0.02661949,  0.9689647 ,
         0.20601757]], dtype=float32)

In [50]:
sim_matrix = clip_images @ clip_texts.T
sim_matrix

array([[ 0.97527176,  0.00725015, -0.02526627,  0.8704373 ,  0.00830222,
         0.00404497],
       [-0.0297686 ,  0.9404535 ,  0.00403179, -0.03243517, -0.14639091,
         0.44930598],
       [-0.01974309,  0.20601763,  0.9689647 , -0.02661949, -0.08548896,
         0.77819383],
       [ 0.6765419 , -0.0191515 , -0.02805255,  0.9236134 , -0.01034794,
        -0.01704965],
       [-0.05516496, -0.08953718, -0.05903826, -0.0396425 ,  0.93743145,
        -0.12680955],
       [ 0.00385709,  0.64253545,  0.6894938 , -0.0119141 , -0.21591164,
         0.97224706]], dtype=float32)

In [51]:
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

{'t2i_precision@1': tensor(1.),
 't2i_recall@1': tensor(1.),
 't2i_ndcg@1': tensor(1.),
 't2i_precision@5': tensor(0.2000),
 't2i_recall@5': tensor(1.),
 't2i_ndcg@5': tensor(1.),
 't2i_precision@10': tensor(0.1667),
 't2i_recall@10': tensor(1.),
 't2i_ndcg@10': tensor(1.),
 'i2t_precision@1': tensor(1.),
 'i2t_recall@1': tensor(1.),
 'i2t_ndcg@1': tensor(1.),
 'i2t_precision@5': tensor(0.2000),
 'i2t_recall@5': tensor(1.),
 'i2t_ndcg@5': tensor(1.),
 'i2t_precision@10': tensor(0.1667),
 'i2t_recall@10': tensor(1.),
 'i2t_ndcg@10': tensor(1.),
 'mean_precision@1': tensor(1.),
 'mean_recall@1': tensor(1.),
 'mean_ndcg@1': tensor(1.),
 'mean_precision@5': tensor(0.2000),
 'mean_recall@5': tensor(1.),
 'mean_ndcg@5': tensor(1.),
 'mean_precision@10': tensor(0.1667),
 'mean_recall@10': tensor(1.),
 'mean_ndcg@10': tensor(1.)}

In [ ]:
#print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    #adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    #print(f"Adjacency matrix shape: {adj_matrix.shape}")
    #sim_matrix, clip_imgs_n, clip_txts_n = get_sim_matrix([t["item1"] for t in loaded_data_new], 
    #                        [t["item2"] for t in loaded_data_new], 
    #                        clip_images, clip_texts,
    #                        img_to_idx, txt_to_idx, out_emb=True)
    
    #idx12idx2 = {}
    #for i, img in enumerate(loaded_data_new):
    #    idx12idx2[i] = img_to_idx[img['item1']]
    #print('idx12idx2', idx12idx2)
    

In [57]:
verbose = True
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    #if typ != 'content':
    #    continue
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    indices_new = [i for i, l in enumerate(loaded_data) if l['link'] == typ]
    adj_matrix = np.eye(len(loaded_data_new)).astype(int)
    # subset of clip_img and clip_texts for loaded_data_new with typ = typ withou img_to_idx and txt_to_idx
    clip_imgs_n = clip_images[indices_new]
    clip_txts_n = clip_texts[indices_new]
    sim_matrix = clip_imgs_n @ clip_txts_n.T
    
    #print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))
    print('uniformity images', uniformity(torch.tensor(clip_imgs_n)))
    print('uniformity texts', uniformity(torch.tensor(clip_txts_n)))
    print('alignment', alignment(torch.tensor(clip_imgs_n), torch.tensor(clip_txts_n)))
    
    if verbose:
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=min(5, len(loaded_data_new)))

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = loaded_data_new[i][query_field]
            recommendations = [loaded_data_new[j]['item2'] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: timeframe
{'t2i_precision@1': tensor(1.), 't2i_recall@1': tensor(1.), 't2i_ndcg@1': tensor(1.), 't2i_precision@5': tensor(1.), 't2i_recall@5': tensor(1.), 't2i_ndcg@5': tensor(1.), 't2i_precision@10': tensor(1.), 't2i_recall@10': tensor(1.), 't2i_ndcg@10': tensor(1.), 'i2t_precision@1': tensor(1.), 'i2t_recall@1': tensor(1.), 'i2t_ndcg@1': tensor(1.), 'i2t_precision@5': tensor(1.), 'i2t_recall@5': tensor(1.), 'i2t_ndcg@5': tensor(1.), 'i2t_precision@10': tensor(1.), 'i2t_recall@10': tensor(1.), 'i2t_ndcg@10': tensor(1.), 'mean_precision@1': tensor(1.), 'mean_recall@1': tensor(1.), 'mean_ndcg@1': tensor(1.), 'mean_precision@5': tensor(1.), 'mean_recall@5': tensor(1.), 'mean_ndcg@5': tensor(1.), 'mean_precision@10': tensor(1.), 'mean_recall@10': tensor(1.), 'mean_ndcg@10': tensor(1.)}
uniformity images tensor(nan)
uniformity texts tensor(nan)
alignment tensor(0.1251)
Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/03766-031madon.jpg
Ground Truth: Artwork pa

### Zero shot classification

In [4]:
annotations = '../SemArt/semart_test.csv'
df = pd.read_csv(annotations, sep='\t', encoding='latin1')
df.shape

(1069, 9)

In [5]:
loaded_data_new = []
for itm in df['IMAGE_FILE']:
    loaded_data_new.append({})
    loaded_data_new[-1]['item1'] = 'Images/' + itm
    author = df[df['IMAGE_FILE'] == itm]['AUTHOR'].values[0]
    loaded_data_new[-1]['author'] = f"Artwork by {author}"
    timeframe = df[df['IMAGE_FILE'] == itm]['TIMEFRAME'].values[0]
    loaded_data_new[-1]['timeframe'] = f"Artwork painted in {timeframe}"
    school = df[df['IMAGE_FILE'] == itm]['SCHOOL'].values[0]
    loaded_data_new[-1]['school'] = f"Artwork from the {school} school"
    material = df[df['IMAGE_FILE'] == itm]['TECHNIQUE'].values[0].split(',')[0]
    loaded_data_new[-1]['material'] = f"Artwork made with {material}"
    genre = df[df['IMAGE_FILE'] == itm]['TYPE'].values[0]
    loaded_data_new[-1]['genre'] = f"Artwork of the {genre} genre"
    loaded_data_new[-1]['link'] = 'metadata'

print(f"Updated loaded_data with authors, total items: {len(loaded_data_new)}")

Updated loaded_data with authors, total items: 1069


In [ ]:
def get_zero_shot_accuracy(item2, verbose=False):
    test_graph_data, _, _ = build_graph_from_json(loaded_data_new, preprocess, tokenizer, 
                                                                           base_folder='../SemArt/', item2=item2)
    test_graph_data = test_graph_data.to(device)
    test_dataset = GraphEdgeDataset(test_graph_data, device=device)
    test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    for batch in test_loader:
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'test')
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_images_cls = F.normalize(embeddings[: len(edge_attr), :], dim=1).cpu().detach().numpy()
        clip_texts_author = F.normalize(embeddings[len(edge_attr): , :], dim=1).cpu().detach().numpy()

        print(f"Extracted {len(clip_texts_author)} text embeddings, each of shape {clip_texts_author[0].shape}")

    adj_matrix_author, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new, field=item2)
    sim_matrix_author = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                                [t[item2] for t in loaded_data_new], 
                                clip_images_cls, clip_texts_author,
                                img_to_idx, txt_to_idx)

      
    if verbose:
        # print(f"Adjacency matrix shape: {adj_matrix_author.shape}", 
        #     f"Similarity matrix shape: {sim_matrix_author.shape}")
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix_author), k=5)

        query_field = 'item1'  # image path
        rec_field = item2      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

        for i, rec_indices in enumerate(recs):
            query = loaded_data_new[i][query_field]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

    return compute_image_to_text_accuracy(sim_matrix_author, adj_matrix_author)
    

In [7]:
for item2 in ['author', 'timeframe', 'school', 'material', 'genre']:
    print(f"Zero-shot classification accuracy for {item2}:")
    acc = get_zero_shot_accuracy(item2)
    print(acc)
    print()

Zero-shot classification accuracy for author:


100%|██████████| 1069/1069 [00:26<00:00, 40.44it/s]
/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Extracted 1069 text embeddings, each of shape (512,)
0.07951356407857811

Zero-shot classification accuracy for timeframe:


100%|██████████| 1069/1069 [00:20<00:00, 51.25it/s]


Extracted 1069 text embeddings, each of shape (512,)
0.13096351730589337

Zero-shot classification accuracy for school:


...

Extracted 1069 text embeddings, each of shape (512,)
0.11973807296538821

Zero-shot classification accuracy for material:


100%|██████████| 1069/1069 [00:18<00:00, 58.34it/s]


Extracted 1069 text embeddings, each of shape (512,)
0.1580916744621141

Zero-shot classification accuracy for genre:


100%|██████████| 1069/1069 [00:17<00:00, 62.71it/s]


Extracted 1069 text embeddings, each of shape (512,)
0.19457436856875585



## Evaluation with pseudo edge index

In [3]:
triplets = '../artistic_sheaf/data/full_triplets.json'
loaded_data = load_json_data(triplets)#[:9914]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

Loaded 9138 triplets from ../artistic_sheaf/data/full_triplets.json


In [4]:
# subselect elts in loaded_data where source == generated_i2t
loaded_data_i2t = [elt for elt in loaded_data if elt['source'] == 'generated_i2t']
print(f"Loaded {len(loaded_data_i2t)} triplets from {triplets} with source generated_i2t")
loaded_data_t2i = [elt for elt in loaded_data if elt['source'] == 'generated_t2i']
print(f"Loaded {len(loaded_data_t2i)} triplets from {triplets} with source generated_t2i")

Loaded 4569 triplets from ../artistic_sheaf/data/full_triplets.json with source generated_i2t
Loaded 4569 triplets from ../artistic_sheaf/data/full_triplets.json with source generated_t2i


In [5]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_i2t, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

100%|██████████| 4569/4569 [00:15<00:00, 286.43it/s] 


4569


/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [6]:
clip_images_i2t = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_images_i2t.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
clip_images_i2t = torch.cat(clip_images_i2t, dim=0)

print(f"Extracted {len(clip_images_i2t)} image embeddings, each of shape {clip_images_i2t[0].shape}")

clip_images_i2t = clip_images_i2t.cpu().detach().numpy()

torch.Size([963, 3, 224, 224]) torch.Size([771, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([612, 3, 224, 224]) torch.Size([507, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([608, 3, 224, 224]) torch.Size([550, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
Extracted 4569 image embeddings, each of shape torch.Size([512])


In [7]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_t2i, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

100%|██████████| 4569/4569 [00:15<00:00, 303.79it/s] 


4569


In [8]:
clip_texts_t2i = []

for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_texts_t2i.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
clip_texts_t2i = torch.cat(clip_texts_t2i, dim=0)

print(f"Extracted {len(clip_texts_t2i)} text embeddings, each of shape {clip_texts_t2i[0].shape}")

clip_texts_t2i = clip_texts_t2i.cpu().detach().numpy()

torch.Size([684, 3, 224, 224]) torch.Size([1488, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([633, 3, 224, 224]) torch.Size([1520, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([538, 3, 224, 224]) torch.Size([1518, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
Extracted 4569 text embeddings, each of shape torch.Size([512])


In [9]:
from collections import defaultdict 
def reorder_predictions_by_link_item(
    predictions: np.ndarray,
    new_list,   # "data/full_triplets.json" (the list used to produce predictions)
    old_list,   # the new file with same links but different item2 assignments
    item = 'item2',  # which item to use for matching (default 'item2' for text predictions)
):
    """
    Reorder the text predictions to match the order of (link, item2) in the new triplet file.
    Assumes:
      - predictions_txt[i] corresponds to old_list[i]['item2'] with old_list[i]['link'].
      - Keys used for matching are (link, item2).
      - Handles duplicate (link, item2) by consuming old indices FIFO.
    """
    
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get(item)
        pos_by_key[(link, item2)].append(idx)

    print(len(pos_by_key), "unique (link,item) pairs in the old file.")
    # Build reorder indices to match new_list order
    
    reorder_indices = []
    missing = []
    for tr in new_list:
        key = (tr.get("link"), tr.get(item))
        if pos_by_key[key]:
            reorder_indices.append(pos_by_key[key].pop(0))  # consume one occurrence
        else:
            missing.append(key)

    if missing:
        # Raise for visibility; switch to a warning if partial overlap is expected.
        example = missing[:5]
        print(
            f"{len(missing)} (link,{item}) pairs in the new file were not found in the old predictions. "
            f"Examples: {example}"
        )

    # Reorder predictions
    idx_t = np.array(reorder_indices)
    print(reorder_indices[:10])
    predictions_reordered = predictions[idx_t]
    
    # print how many triplets (link, item1, item2) are the same in the old and new list by creating dictionaries
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key_all = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get('item2')
        item1 = tr.get('item1')
        pos_by_key_all[(link, item2, item1)].append(idx)

    wrong = []
    for tr in new_list:
        key = (tr.get("link"), tr.get('item2'), tr.get('item1'))
        if pos_by_key_all[key]:
            reorder_indices.append(pos_by_key_all[key].pop(0))  # consume one occurrence
        else:
            wrong.append(key)

    print("Accuracy of preliminary matching (link, item1, item2):",
          1 - len(wrong) / len(new_list))

    return predictions_reordered


In [10]:
loaded_data = load_json_data("data/triplets_semart_test_orig.json")#[:9914]
print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_orig.json")

Loaded 4569 triplets from data/triplets_semart_test_orig.json


In [11]:
predictions_txt_new_order = reorder_predictions_by_link_item(
    clip_texts_t2i,
    new_list=loaded_data_t2i,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item2'
)
print(predictions_txt_new_order.shape)
predictions_txt_new_order[:5]
    

4546 unique (link,item) pairs in the old file.
[0, 8, 12, 22, 24, 28, 32, 36, 40, 44]
Accuracy of preliminary matching (link, item1, item2): 0.24403589406872406
(4569, 512)


array([[-0.0004141 ,  0.04339724,  0.05340251, ..., -0.01926412,
         0.00413403,  0.01731746],
       [-0.04599962, -0.01825658,  0.02921375, ..., -0.06506981,
        -0.01110992,  0.05403557],
       [-0.01743414,  0.05956129, -0.05097187, ..., -0.03625222,
         0.02499957, -0.05406392],
       [-0.05788532, -0.01635166, -0.06560384, ...,  0.01230823,
        -0.03691173, -0.03298913],
       [-0.00270245, -0.0558102 ,  0.01303499, ..., -0.05708861,
        -0.04821299,  0.00346222]], shape=(5, 512), dtype=float32)

In [12]:
predictions_image_new_order = reorder_predictions_by_link_item(
    clip_images_i2t,
    new_list=loaded_data_i2t,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item1',
)
print(predictions_image_new_order.shape)

2520 unique (link,item) pairs in the old file.
[0, 8, 12, 22, 24, 28, 32, 36, 40, 44]
Accuracy of preliminary matching (link, item1, item2): 0.15364412344057776
(4569, 512)


In [13]:
verbose = False
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t["item2"] for t in loaded_data_new], 
                            clip_images_i2t, clip_texts_t2i,
                            img_to_idx, txt_to_idx)
    print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))

    if verbose:
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=5)

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = loaded_data_new[i][query_field]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: context
Loaded 1453 triplets for type context
Adjacency matrix shape: (634, 1447)
Similarity matrix shape: (634, 1447)
{'t2i_precision@1': tensor(0.1088), 't2i_recall@1': tensor(0.0685), 't2i_ndcg@1': tensor(0.1088), 't2i_precision@5': tensor(0.0356), 't2i_recall@5': tensor(0.1088), 't2i_ndcg@5': tensor(0.1015), 't2i_precision@10': tensor(0.0205), 't2i_recall@10': tensor(0.1237), 't2i_ndcg@10': tensor(0.1065), 'i2t_precision@1': tensor(0.0560), 'i2t_recall@1': tensor(0.0560), 'i2t_ndcg@1': tensor(0.0560), 'i2t_precision@5': tensor(0.0182), 'i2t_recall@5': tensor(0.0909), 'i2t_ndcg@5': tensor(0.0757), 'i2t_precision@10': tensor(0.0112), 'i2t_recall@10': tensor(0.1116), 'i2t_ndcg@10': tensor(0.0817), 'mean_precision@1': tensor(0.0824), 'mean_recall@1': tensor(0.0622), 'mean_ndcg@1': tensor(0.0824), 'mean_precision@5': tensor(0.0269), 'mean_recall@5': tensor(0.0998), 'mean_ndcg@5': tensor(0.0886), 'mean_precision@10': tensor(0.0159), 'mean_recall@10': tensor(0.1176), 'mea